<a href="https://colab.research.google.com/github/koushikkkkkkkkkk/prostate-mil-project/blob/main/notebooks/05_transmil_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 - Simplified TransMIL Training on Real PANDA Data

**Role:** Person 3 (Himesh)
**Model:** TransMIL (Transformer-based Correlated Multiple Instance Learning)
**Task:** Train the TransMIL model on real PANDA features across 3 seeds (0, 1, 2) and evaluate against the baseline.


## 1. Setup & Environment (Colab & Local Compatible)
Mounts Google Drive where features are saved and sets the repository root.

In [ ]:
import os, sys, json
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    if not os.path.exists('prostate-mil-project'):
        !git clone https://github.com/koushikkkkkkkkkk/prostate-mil-project.git
        %cd prostate-mil-project
    else:
        %cd prostate-mil-project
        !git pull origin main
    !pip install -q pyyaml
except ImportError:
    IN_COLAB = False

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")
import numpy as np, pandas as pd, torch
print(f"Working Directory: {Path.cwd()}")
print(f"PyTorch: {torch.__version__} | GPU Available: {torch.cuda.is_available()}")

## 2. Locate Data & Link Google Drive Features

In [ ]:
MANIFEST = ROOT / "data" / "common_split.csv"

# Search Google Drive paths or local path for pre-extracted features
DRIVE_FEATURES_CANDIDATES = [
    Path("/content/drive/MyDrive/panda_mil_project/features"),
    Path("/content/drive/MyDrive/panda_mil_data/features"),
    Path("/content/drive/MyDrive/prostate_mil/features")
]

LOCAL_FEATURES = ROOT / "data" / "features"
if not LOCAL_FEATURES.exists():
    for candidate in DRIVE_FEATURES_CANDIDATES:
        if candidate.exists():
            LOCAL_FEATURES.symlink_to(candidate)
            print(f"Symlinked {candidate} -> {LOCAL_FEATURES}")
            break

FEATURES = LOCAL_FEATURES if LOCAL_FEATURES.exists() else ROOT / "data" / "features"
CONFIG = ROOT / "configs" / "transmil.yaml"
OUT = ROOT / "results" / "metrics" / "transmil"

print("Manifest:", MANIFEST, "(exists:", MANIFEST.exists(), ")")
print("Features:", FEATURES, "(exists:", FEATURES.exists(), ")")
print("Config:  ", CONFIG, "(exists:", CONFIG.exists(), ")")
print("Output:  ", OUT)

## 3. Model Sanity Check: Shapes & Parameters

In [ ]:
import yaml
from src.models import build_model, count_parameters

cfg = yaml.safe_load(CONFIG.read_text())["model"]
cfg["in_dim"] = 1024
model = build_model("transmil", cfg)
model.eval()

dummy_bag = torch.randn(1, 64, cfg["in_dim"])
with torch.no_grad():
    out = model(dummy_bag)

print("Logits shape:   ", tuple(out["logits"].shape))
print("Attention shape:", tuple(out["attention"].shape))
print("Parameters:     ", f"{count_parameters(model):,}")
print("Attention sum:  ", float(out["attention"].sum()))

## 4. Train TransMIL (3 Seeds: 0, 1, 2)
Trains on GPU with early stopping based on validation QWK.

In [ ]:
import subprocess, sys

seeds = ["0", "1", "2"]
cmd = [sys.executable, "-m", "src.train", "--model", "transmil",
       "--manifest", str(MANIFEST), "--features-root", str(FEATURES),
       "--config", str(CONFIG), "--out-dir", str(OUT), "--seeds", *seeds]

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT)

print(f"Executing: {' '.join(cmd)}")
res = subprocess.run(cmd, cwd=ROOT, env=env, capture_output=True, text=True)
print(res.stdout)

if res.returncode != 0:
    print("--- ERROR (STDERR) ---")
    print(res.stderr)
    raise RuntimeError(f"Training exited with code {res.returncode}")
else:
    print("\nSUCCESS: TransMIL training complete across all 3 seeds!")

## 5. Review Official TransMIL Results

In [ ]:
import json

agg_path = OUT / "aggregate_transmil.json"
if agg_path.exists():
    agg = json.loads(agg_path.read_text())
    print("====================================================")
    print(f"TransMIL Mean Test QWK: {agg['qwk']['mean']:.4f} ± {agg['qwk']['std']:.4f}")
    print(f"Accuracy:              {agg['accuracy']['mean']:.4f}")
    print(f"Balanced Accuracy:     {agg['balanced_accuracy']['mean']:.4f}")
    print(f"Macro F1:              {agg['macro_f1']['mean']:.4f}")
    print("====================================================")
else:
    for seed in ["0", "1", "2"]:
        m_path = OUT / f"metrics_transmil_seed{seed}.json"
        if m_path.exists():
            m = json.loads(m_path.read_text())
            print(f"Seed {seed}: QWK = {m.get('test_qwk', m.get('qwk')):.4f}")

table_path = OUT / "table_transmil.md"
if table_path.exists():
    print("\n" + table_path.read_text())

## 6. Attention Behaviour & Entropy

In [ ]:
with torch.no_grad():
    out = model(torch.randn(1, 400, 1024))
a = out["attention"].squeeze(0).cpu().numpy()
entropy = float(-(a * np.log(a + 1e-12)).sum())
top10_share = float(np.sort(a)[-10:].sum())
print(f"Attention Entropy: {entropy:.4f} | Max Attention: {float(a.max()):.6f} | Top-10 Mass: {top10_share:.4f}")